# Delta Sharing: explore and visualise your shared data

This notebook shows how to access the data shared with you and turn it into charts. It works with any shared table: everything specific to your setup (Keycloak login, server URL, default table, partition column) is read from the `.env` file.

You'll:
1. log in with Keycloak, in the browser or with a service account;
2. find every table you can access and inspect its columns, without downloading data;
3. load a sample and profile it;
4. load only the time range you need, using the table's partition column;
5. chart the data (trends, top values, distributions, data quality);
6. compare tables, read earlier versions and export to Parquet, CSV or Excel.

**Before you start:** `pip install delta-sharing-sso[notebook]`, copy `.env.example` to `.env` and fill it in, then check your settings with `delta-sharing-sso-check`. Then run the cells from top to bottom with Shift+Enter.

In [ ]:
import pandas as pd
import plotly.express as px

from delta_sharing_sso import Settings, create_client
from delta_sharing_sso.explore_helpers import (add_partition_datetime, choose_partition, coerce_datetimes,
                                               column_roles, distribution, measure_over_time, null_heatmap,
                                               profile, rows_over_time, top_values)
from delta_sharing_sso.predicates import equal, time_equal, time_range

pd.set_option("display.max_columns", 50)
settings = Settings.from_env()
settings

# 1. Log in

Uses `AUTH_METHOD` from `.env`:
- `keycloak` opens your browser for the Keycloak login. Complete it there and come back. When the token expires it's renewed silently; the browser only opens again once your Keycloak session has ended.
- `keycloak_service_account` logs in with the client secret in `.env`, with no browser. Tokens are renewed automatically, which suits scheduled jobs.

In [ ]:
catalog, login = create_client(settings)
print("Login method:", settings.auth_description)
login.get_access_token()
print("Logged in. Token valid until:", login.expiry_info().get("access_expires_at", "unknown"))

# 2. Find the tables you can access
Data is organised as **share → schema → table**.

In [ ]:
tables = catalog.tables_overview()
print(f"{len(tables)} tables available")
tables

### Browse tables like a bucket (optional)

Instead of tracking `share`, `schema`, `table` as three separate strings, you can browse `catalog` directly — it behaves like a folder tree, and tab-completes in Jupyter. This is equivalent to the explicit calls above and used purely for convenience.

In [ ]:
# catalog.<share>.<table>            (if the table name is unique within that share)
# catalog.<share>.<schema>.<table>   (always works)
# catalog["share.schema.table"]      (bracket form, e.g. for names with dots or as a variable)
# catalog.find("orders")             -> full names containing "orders"
#
# Try tab-completion: catalog.<Tab>, then catalog.<share>.<Tab>
print("Shares you can tab-complete into:", sorted(tables["share"].unique()))
print("Tables matching 'a':", catalog.find("a")[:5])

# 3. Inspect a table

The table comes from `DEFAULT_TABLE` in `.env`, or is the first one found. To look at another, set `TABLE_NAME` to a `full_name` from the list above. Columns, partition columns and version come from metadata, so no data is downloaded.

In [ ]:
TABLE_NAME = settings.default_table or tables["full_name"].iloc[0]   # <- or e.g. "my_share.my_schema.my_table"
SHARE, SCHEMA, TABLE = TABLE_NAME.split(".", 2)

columns = catalog.get_table_columns(SHARE, SCHEMA, TABLE)
version = catalog.get_table_version(SHARE, SCHEMA, TABLE)
partition_columns = columns.loc[columns["partition_column"], "name"].tolist()
print(f"{TABLE_NAME}: version {version}, {len(columns)} columns, partitioned by {partition_columns or 'nothing'}")
columns

# 4. Load a sample and profile it
A row limit keeps this fast and is enough to understand the data.

In [ ]:
sample_raw = catalog.load_table_as_dataframe(SHARE, SCHEMA, TABLE, limit=1000)
sample = coerce_datetimes(sample_raw)
print(f"{len(sample):,} rows x {sample.shape[1]} columns")
sample.head()

In [ ]:
profile(sample)

## How is the table partitioned?

Filtering on a partition column lets the server skip whole files, so you download far less. The cell below picks the partition column (or `PARTITION_COLUMN` from `.env`) and works out how it stores time: epoch seconds or milliseconds, a date, a timestamp, or a date string. If the guess is wrong, set `PARTITION_FORMAT` in `.env`.

In [ ]:
partition = choose_partition(columns, sample_raw, column=settings.partition_column, fmt=settings.partition_format,
                             value_type=settings.partition_value_type,
                             string_format=settings.partition_string_format)
if partition is None:
    print("This table has no partition column, so filters are applied after loading.")
elif partition.is_time:
    print(f"Time partition: {partition.column} (stored as {partition.fmt}, predicate type {partition.value_type})")
    sample = add_partition_datetime(coerce_datetimes(sample_raw, exclude=[partition.column]), partition)
    print("Sample covers", sample["partition_time"].min(), "to", sample["partition_time"].max())
else:
    print(f"Partition column {partition.column} isn't time-based. "
          f"Example values: {sample[partition.column].dropna().unique()[:5].tolist()}")

# 5. Load only what you need

For a time partition, set `START` and `END` (inclusive, UTC). For another kind of partition, set `PARTITION_VALUE` to the value you want. `MAX_ROWS` is a safety limit; set it to `None` to load everything that matches.

Predicate hints are hints only, so the cell filters again in pandas to guarantee exact results.

In [ ]:
START, END = "2026-05-01", "2026-05-31"   # <- for time partitions
PARTITION_VALUE = None                     # <- for other partitions, e.g. "DE"
MAX_ROWS = 200_000

hints = None
if partition is not None and partition.is_time:
    hints = time_range(partition.column, START, END, partition.fmt, partition.value_type, partition.string_format)
elif partition is not None and PARTITION_VALUE is not None:
    hints = equal(partition.column, PARTITION_VALUE, partition.value_type)

df = catalog.load_table_as_dataframe(SHARE, SCHEMA, TABLE, limit=MAX_ROWS, json_predicate_hints=hints)
df = add_partition_datetime(coerce_datetimes(df, exclude=[partition.column] if partition else []), partition)

if "partition_time" in df.columns:
    end_of_day = pd.Timestamp(END, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)
    df = df[df["partition_time"].between(pd.Timestamp(START, tz="UTC"), end_of_day)]
    print(f"{len(df):,} rows between {START} and {END}")
elif hints is not None:
    df = df[df[partition.column].astype(str) == str(PARTITION_VALUE)]
    print(f"{len(df):,} rows where {partition.column} = {PARTITION_VALUE}")
else:
    print(f"{len(df):,} rows loaded (no partition filter)")
if MAX_ROWS and len(df) >= MAX_ROWS:
    print(f"Note: the MAX_ROWS limit of {MAX_ROWS:,} was reached, so this may not be all the data.")

# 6. Visualise

The helpers detect which columns are dates, measures and categories, so these charts work for any table. Charts are interactive: hover for values, drag to zoom, double-click to reset.

In [ ]:
roles = column_roles(df, exclude=[partition.column] if partition else [])
DATE_COL = "partition_time" if "partition_time" in df.columns else next(iter(roles["datetimes"]), None)
CATEGORY = next(iter(roles["categories"]), None)   # <- or set a column name
MEASURE = next(iter(roles["measures"]), None)      # <- or set a column name
print("Date column:", DATE_COL, "| category:", CATEGORY, "| measure:", MEASURE)
roles

## Rows over time

In [ ]:
if DATE_COL:
    rows_over_time(df, DATE_COL, freq="D", title="Rows per day").show()
else:
    print("No date or time column found in this table.")

## Top values of a category

In [ ]:
if CATEGORY:
    top_values(df, CATEGORY, top_n=15).show()
else:
    print("No low-cardinality text column found.")

## A measure over time, and its distribution

In [ ]:
if MEASURE and DATE_COL:
    measure_over_time(df, DATE_COL, MEASURE, agg="sum", freq="D").show()
if MEASURE:
    distribution(df, MEASURE, color=CATEGORY).show()
else:
    print("No numeric measure found in this table.")

## Data quality: missing values

In [ ]:
fig = null_heatmap(df)
fig.show() if fig else print("No missing values.")

## Build your own chart

Use Plotly Express directly with the columns you care about. Useful chart types: `px.bar`, `px.line`, `px.scatter`, `px.histogram`, `px.box`, `px.treemap`.

In [ ]:
# Example: rows per category over time. Replace the column names with ones from your table.
if DATE_COL and CATEGORY:
    per_day = df.groupby([pd.Grouper(key=DATE_COL, freq="D"), CATEGORY]).size().reset_index(name="rows")
    px.bar(per_day, x=DATE_COL, y="rows", color=CATEGORY, title=f"Rows per day by {CATEGORY}").show()

# 7. Compare tables

Load the same time range from several tables and compare their volume. The tables come from `COMPARE_TABLES` in `.env`, or you can list them here. Each needs a time partition column.

In [ ]:
COMPARE = settings.compare_tables or []   # <- e.g. ["share.schema.table_a", "share.schema.table_b"]

frames = []
for name in COMPARE:
    try:
        s, sc, t = name.split(".", 2)
        cols = catalog.get_table_columns(s, sc, t)
        smp = catalog.load_table_as_dataframe(s, sc, t, limit=200)
        part = choose_partition(cols, smp, fmt=settings.partition_format,
                                value_type=settings.partition_value_type,
                                string_format=settings.partition_string_format)
        if part is None or not part.is_time:
            print(f"Skipped {name}: no time partition column")
            continue
        data = catalog.load_table_as_dataframe(
            s, sc, t, limit=MAX_ROWS,
            json_predicate_hints=time_range(part.column, START, END, part.fmt, part.value_type, part.string_format))
        data = add_partition_datetime(data, part)
        data = data[data["partition_time"].between(pd.Timestamp(START, tz="UTC"),
                                                   pd.Timestamp(END, tz="UTC") + pd.Timedelta(days=1))]
        frames.append(data.groupby(pd.Grouper(key="partition_time", freq="D")).size()
                          .reset_index(name="rows").assign(table=name))
    except Exception as e:
        print(f"Skipped {name}: {e}")

if frames:
    px.line(pd.concat(frames), x="partition_time", y="rows", color="table", markers=True,
            title=f"Rows per day, {START} to {END}").show()
elif not COMPARE:
    print("Set COMPARE_TABLES in .env or list tables in COMPARE to compare them.")

# 8. Earlier versions (time travel)

Each update to a shared table creates a new version. If history is shared with you, you can reload an earlier version, for example to reproduce a previous report.

In [ ]:
previous_version = max(version - 1, 0)
try:
    previous = catalog.load_table_as_dataframe(SHARE, SCHEMA, TABLE, limit=1000, version=previous_version)
    print(f"Version {previous_version}: loaded {len(previous):,} sample rows (current version is {version})")
except Exception as e:
    print("Earlier versions aren't available for this table:", e)

# 9. Export
Save the filtered data for Excel, Power BI or other tools.

In [ ]:
out = TABLE_NAME.replace(".", "_") + (f"_{START}_{END}" if "partition_time" in df.columns else "")
export = df.copy()
for col in export.select_dtypes("datetimetz").columns:   # Excel can't store time zones
    export[col] = export[col].dt.tz_localize(None)

export.to_parquet(f"{out}.parquet", index=False)   # compact and keeps column types
export.to_csv(f"{out}.csv", index=False)
if len(export) < 1_000_000:                         # Excel's sheet limit is about 1M rows
    export.to_excel(f"{out}.xlsx", index=False)
print("Saved", out)

# Troubleshooting

| Symptom | Likely cause and fix |
|---|---|
| `AUTH_METHOD=... is not supported` | Use `keycloak` or `keycloak_service_account` in `.env`. |
| `ValueError: server_url is required` / `client_id is required` | `KEYCLOAK_URL` or `KEYCLOAK_CLIENT_ID` is missing from `.env`. |
| `Set DELTA_SHARING_URL` | `DELTA_SHARING_URL` is missing from `.env`. |
| `TimeoutError: No authorization code received` | The browser login wasn't completed in time. Run the login cell again. |
| `OSError: [Errno 98] Address already in use` | Port 8000 (the login callback) is busy. Free it, or set `KEYCLOAK_REDIRECT_URI` to another localhost port allowed in the Keycloak client. |
| `401: unauthorized_client` / `invalid_client_credentials` | Wrong `KEYCLOAK_CLIENT_SECRET`, or service accounts aren't enabled for the Keycloak client. |
| `401` / `403` from the Delta Sharing server | Login worked but your account has no access to the share yet. Contact your data provider. |
| `SSL: CERTIFICATE_VERIFY_FAILED` | Your CA certificate isn't in the `certifi` bundle. See *Certificate configuration* in `README.md`. |
| Filter returns no or too many rows | The partition format was guessed wrongly. Set `PARTITION_FORMAT` (and if needed `PARTITION_VALUE_TYPE`) in `.env`. |
| Out of memory | Narrow the time range in section 5, or keep `MAX_ROWS` set. |